<a href="https://colab.research.google.com/github/LanovenkoOleh/ML/blob/main/%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%9B%D0%B0%D0%BD%D0%BE%D0%B2%D0%B5%D0%BD%D0%BA%D0%BE_%D0%A4%D0%86%D0%A2_3_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Лановенко Олег Артемович

**Група:** ФІТ 3-10

**Дата:** 04.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [32]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [33]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [34]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [35]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
print('Кількість користувачів:', ratings_df['user_id'].nunique())

print('Кількість фільмів:', ratings_df['item_id'].nunique())

print('Мінімальна оцінка:', ratings_df['rating'].min())
print('Максимальна оцінка:', ratings_df['rating'].max())
print('Середня оцінка:', ratings_df['rating'].mean())

print('\nПропущені значення:')
print(ratings_df.isnull().sum())

print('\nРозподіл оцінок:')
print(ratings_df['rating'].value_counts().sort_index())

Кількість користувачів: 943
Кількість фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.52986

Пропущені значення:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64

Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64


**Короткі спостереження щодо даних:**

Набір даних MovieLens 100K містить 100 000 оцінок, виставлених 943 користувачами для 1682 фільмів. Оцінки представлені за п'ятибальною шкалою від 1 до 5, причому середнє значення становить приблизно 3,53. Пропущених значень немає, а найчастіше користувачі виставляють оцінку 4.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [36]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print('Навчальна вибірка:', train_df.shape)
print('Тестова вибірка:', test_df.shape)

Навчальна вибірка: (80000, 4)
Тестова вибірка: (20000, 4)


In [37]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(
    rating_scale=data.reader.rating_scale
)



train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)


testset = list(
    test_df[['user_id', 'item_id', 'rating']]
    .itertuples(index=False, name=None)
)

## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [38]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

Для моделі SVD обрано параметри n_factors (50 і 100), які визначають кількість прихованих факторів, lr_all (0.005 і 0.010), що впливає на швидкість навчання, та reg_all (0.02 і 0.05), який регулює перенавчання. Для KNNBasic перевіряється кількість сусідів k (20 і 40), мінімальна кількість сусідів min_k (1 і 5), а також метрики схожості cosine та pearson. Додатково порівнюються підходи на основі користувачів і фільмів. Обрані значення дозволяють дослідити вплив різних налаштувань на точність рекомендацій.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [39]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:

grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_svd.fit(train_data)

In [ ]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:

grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_knn.fit(train_data)

In [ ]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:

print('\nНайкращий результат SVD:')
print('RMSE:', grid_search_svd.best_score['rmse'])
print('MAE:', grid_search_svd.best_score['mae'])
print('Параметри за RMSE:', grid_search_svd.best_params['rmse'])

print('\nНайкращий результат KNNBasic:')
print('RMSE:', grid_search_knn.best_score['rmse'])
print('MAE:', grid_search_knn.best_score['mae'])
print('Параметри за RMSE:', grid_search_knn.best_params['rmse'])

**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | ... | ... | ... |
| KNNBasic | ... | ... | ... |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

In [ ]:
cv_results = pd.DataFrame({
    'Модель': ['SVD', 'KNNBasic'],
    'CV RMSE': [
        grid_search_svd.best_score['rmse'],
        grid_search_knn.best_score['rmse']
    ],
    'CV MAE': [
        grid_search_svd.best_score['mae'],
        grid_search_knn.best_score['mae']
    ],
    'Найкращі параметри за RMSE': [
        str(grid_search_svd.best_params['rmse']),
        str(grid_search_knn.best_params['rmse'])
    ]
})

display(cv_results)

За результатами трикратної кросвалідації було визначено оптимальні гіперпараметри моделей SVD та KNNBasic. Основним критерієм порівняння обрано RMSE, оскільки ця метрика враховує величину помилок прогнозування оцінок. Модель із меншим значенням CV RMSE вважається кращою за результатами перехресної перевірки та буде використана для формування рекомендацій.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [ ]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:

best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']


trainset = train_data.build_full_trainset()


best_svd = SVD(
    **best_svd_params,
    random_state=42
)

best_svd.fit(trainset)


best_knn = KNNBasic(
    **best_knn_params
)

best_knn.fit(trainset)

## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [ ]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:

svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)


print('\nРезультати SVD на тестовій вибірці:')
svd_rmse = accuracy.rmse(
    svd_predictions,
    verbose=True
)
svd_mae = accuracy.mae(
    svd_predictions,
    verbose=True
)


print('\nРезультати KNNBasic на тестовій вибірці:')
knn_rmse = accuracy.rmse(
    knn_predictions,
    verbose=True
)
knn_mae = accuracy.mae(
    knn_predictions,
    verbose=True
)

In [ ]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
}).sort_values('RMSE')

print('\nПорівняння моделей:')
display(results)


best_model_name = results.iloc[0]['Model']

print(
    f"Найкраща модель за RMSE: {best_model_name}"
)

Після навчання моделей було проведено оцінювання на незалежній тестовій вибірці за метриками RMSE та MAE. Отримані результати дозволяють оцінити точність прогнозування рейтингів фільмів на даних, які не використовувалися під час навчання. Порівняння з результатами кросвалідації дає змогу визначити, наскільки стабільно моделі працюють на нових даних. Незначна різниця між CV та тестовими показниками свідчитиме про достатню узагальнювальну здатність моделей.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [ ]:
# Початковий ID можна змінити

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:

if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model = best_svd
    best_model_name = 'SVD'
else:
    best_model = best_knn
    best_model_name = 'KNNBasic'

print(f"Для рекомендацій використовується модель: {best_model_name}")


user_id = '196'

try:
    inner_user_id = trainset.to_inner_uid(user_id)

    user_ratings = trainset.ur[inner_user_id]

    print(
        f"Кількість оцінених фільмів користувачем "
        f"{user_id}: {len(user_ratings)}"
    )

    all_inner_items = set(trainset.all_items())

    rated_inner_items = {
        inner_item_id
        for inner_item_id, rating in user_ratings
    }

    unrated_inner_items = all_inner_items - rated_inner_items

    recommendations = []

    for inner_item_id in unrated_inner_items:
        raw_item_id = trainset.to_raw_iid(inner_item_id)

        predicted_rating = best_model.predict(
            uid=user_id,
            iid=raw_item_id
        ).est

        recommendations.append({
            'item_id': raw_item_id,
            'predicted_rating': predicted_rating
        })

    recommendations_df = (
        pd.DataFrame(recommendations)
        .sort_values(
            by='predicted_rating',
            ascending=False
        )
        .head(10)
        .reset_index(drop=True)
    )

    print(
        f"\nТоп-10 рекомендацій для користувача {user_id}:"
    )

    display(recommendations_df)

except ValueError:
    print(
        f"Користувача з ID {user_id} "
        "немає в навчальній вибірці."
    )

Для користувача з ID 196 було сформовано список із десяти рекомендованих фільмів на основі прогнозованих рейтингів. Для цього використано модель, яка продемонструвала найменше значення RMSE під час кросвалідації. Із рекомендацій виключено фільми, які користувач уже оцінив у навчальній вибірці. Отриманий список відсортовано за спаданням прогнозованих оцінок, що дозволяє першими відображати найбільш перспективні рекомендації.

## 9. Перевірка виконання

Позначте після реалізації:

- [+] Дані `ml-100k` завантажено та перевірено.
- [+] Є розбиття 80/20 з фіксованим `random_state`.
- [+] `Reader`, `train_data` і `testset` створені коректно.
- [+] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [+] Для SVD і KNNBasic показано параметри та результати CV.
- [+] Моделі навчено на повній навчальній частині.
- [+] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [+] Наведено порівняльну таблицю та інтерпретацію.
- [+] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [+] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---## Висновки

У ході виконання лабораторної роботи було досліджено принципи побудови рекомендаційних систем із використанням алгоритмів SVD та KNNBasic на наборі даних MovieLens 100K, який містить 100 000 оцінок від 943 користувачів для 1682 фільмів. Було проведено попередній аналіз даних, розбиття на навчальну (80%) та тестову (20%) вибірки, а також підбір оптимальних гіперпараметрів методом GridSearchCV із трикратною кросвалідацією. За результатами кросвалідації найкращою виявилася модель **SVD із CV RMSE = 0,9451 та CV MAE = 0,7455**, для якої оптимальними параметрами стали n_factors = 50, lr_all = 0,01 та reg_all = 0,05, тоді як для KNNBasic найкращими параметрами були k = 40, min_k = 1, метрика схожості cosine та user_based = True із CV RMSE = 1,0276 і CV MAE = 0,8153. На незалежній тестовій вибірці модель **SVD отримала RMSE = 0,9192 та MAE = 0,7234**, тоді як KNNBasic продемонструвала RMSE = 1,0120 та MAE = 0,7984, що підтвердило вищу точність SVD. Порівняння результатів кросвалідації та тестування показало незначне зменшення похибок на тестових даних для обох моделей, що свідчить про стабільність їхньої роботи та відсутність очевидних ознак перенавчання за цими показниками. На основі найкращої моделі SVD було сформовано топ-10 рекомендацій для користувача з ID 196, який мав 29 оцінених фільмів у навчальній вибірці, причому найвища прогнозована оцінка становила 4,4904 для фільму з ID 320. Основними обмеженнями рекомендаційної системи залишаються проблема холодного старту для нових користувачів і фільмів, а також залежність точності прогнозування від кількості доступних оцінок. Таким чином, у роботі було практично реалізовано повний процес створення, налаштування та оцінювання рекомендаційної системи, а також встановлено, що **алгоритм SVD є ефективнішим за KNNBasic для дослідженого набору даних та обраних налаштувань**.